# CLIP Image Search — Google Colab

This notebook builds a semantic image search engine using OpenAI CLIP.
- **Text search**: find images matching a natural language query
- **Image search**: find images similar to a query image

> **Runtime**: Go to `Runtime → Change runtime type → T4 GPU` for faster embedding.


## Step 1 — Install packages

We pin `transformers<4.46` to avoid a breaking change where `get_image_features()` returns a dataclass instead of a plain tensor.


In [ ]:
!pip install -q \
    'transformers>=4.35.0,<4.46.0' \
    'faiss-cpu>=1.7.4' \
    'Pillow>=10.0.0' \
    'tqdm>=4.65.0' \
    'torch>=2.0.0' \
    'torchvision>=0.15.0' \
    'matplotlib>=3.7.0' \
    'scikit-learn>=1.3.0'

print('Packages installed.')

## Step 2 — Clone your repo from GitHub

Replace `YOUR_USERNAME` and `YOUR_REPO` with your actual GitHub details.
If you already have the files uploaded to Colab or on Drive, skip this cell.


In [ ]:
import os

GITHUB_USER = 'ridhiman7'          # your GitHub username
GITHUB_REPO = 'YOUR_REPO_NAME'     # <-- REPLACE with your actual repo name
REPO_DIR    = f'/content/{GITHUB_REPO}'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git
else:
    print(f'Repo already at {REPO_DIR}, pulling latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## Step 3 — Imports


In [ ]:
import sys, os
sys.path.insert(0, '.')   # make local modules importable

import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image

from embedding import CLIPEmbedder
from index import FaissIndex
from search import ImageSearchEngine

IMAGE_DIR = './images'
INDEX_DIR = './index'

print('All imports OK.')

## Step 4 — Provide images

**Option A** — Use your own images (upload a zip to Colab Files, then unzip):
```python
!unzip /content/my_images.zip -d ./images
```

**Option B** — Auto-generate 30 synthetic coloured demo images (runs automatically if `./images` is empty).


In [ ]:
def create_demo_images(folder: str = IMAGE_DIR, n: int = 30):
    """Generate synthetic solid-colour images for a quick smoke-test."""
    Path(folder).mkdir(parents=True, exist_ok=True)
    color_map = {
        'red':    (220, 50,  50),
        'blue':   (50,  100, 220),
        'green':  (50,  180, 80),
        'yellow': (240, 210, 50),
        'purple': (150, 50,  200),
    }
    for i in range(n):
        name = random.choice(list(color_map.keys()))
        canvas = np.full((224, 224, 3), color_map[name], dtype=np.uint8)
        Image.fromarray(canvas).save(Path(folder) / f'{name}_{i:03d}.jpg')
    print(f'Created {n} demo images in "{folder}".')

# --- Run Option A first (upload/unzip), or fall back to demo images ---
existing = list(Path(IMAGE_DIR).glob('*.*')) if Path(IMAGE_DIR).exists() else []
img_exts = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
existing_imgs = [p for p in existing if p.suffix.lower() in img_exts]

if not existing_imgs:
    print('No images found — generating demo images...')
    create_demo_images()
else:
    print(f'Found {len(existing_imgs)} images in "{IMAGE_DIR}".')

## Step 5 — Build (or load) the search index

This step:
1. Loads the CLIP model (`openai/clip-vit-base-patch32`, ~350 MB)
2. Encodes all images into 512-d embeddings
3. Builds a FAISS flat inner-product index
4. Saves the index to `./index/`

Re-running the notebook will **load** the saved index instead of rebuilding.


In [ ]:
if Path(INDEX_DIR).exists() and list(Path(INDEX_DIR).glob('*')):
    print(f'Loading existing index from "{INDEX_DIR}"...')
    engine = ImageSearchEngine.load(INDEX_DIR)
else:
    print(f'Building new index from "{IMAGE_DIR}"...')
    engine = ImageSearchEngine.build_from_folder(
        image_dir=IMAGE_DIR,
        index_dir=INDEX_DIR,
        index_type='flat',   # exact search; use 'ivf' for 100k+ images
        batch_size=32,
    )

print(f'Index ready: {engine.index_size()} images indexed.')

## Step 6 — Helper: display results inline


In [ ]:
def show_results(results, query_label: str, max_cols: int = 5):
    """Render a result grid inline in the notebook."""
    n = len(results)
    if n == 0:
        print('No results.')
        return

    cols = min(n, max_cols)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows + 0.5))
    fig.suptitle(f'Query: "{query_label}"', fontsize=13, fontweight='bold')

    flat_axes = np.array(axes).flatten() if n > 1 else [axes]

    for ax, result in zip(flat_axes, results):
        try:
            img = mpimg.imread(result.image_path)
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, 'Error loading', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(
            f'#{result.rank}  score={result.score:.3f}\n{Path(result.image_path).name}',
            fontsize=7,
        )
        ax.axis('off')

    for ax in flat_axes[n:]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

print('show_results() ready.')

## Step 7 — Text-to-image search

Edit `QUERY` to search for anything.


In [ ]:
QUERY = 'a red object'   # <-- change this
K     = 5                # number of results

results = engine.text_search(QUERY, k=K)
show_results(results, QUERY)

### Run several queries at once


In [ ]:
queries = [
    'a red object',
    'something blue',
    'a green image',
    'yellow',
    'purple',
]

for q in queries:
    results = engine.text_search(q, k=5)
    show_results(results, q)

## Step 8 — Image-to-image search (optional)

Upload a query image and find visually/semantically similar images in the index.


In [ ]:
from google.colab import files

print('Upload a query image:')
uploaded = files.upload()

if uploaded:
    query_path = list(uploaded.keys())[0]
    print(f'Using "{query_path}" as query...')
    results = engine.image_search(query_path, k=5)
    show_results(results, f'Image: {query_path}')
else:
    print('No file uploaded.')

## Step 9 — Rebuild index from scratch (optional)

Run this if you add new images to `./images/` and want to refresh the index.


In [ ]:
import shutil

# Remove old index
if Path(INDEX_DIR).exists():
    shutil.rmtree(INDEX_DIR)
    print('Old index removed.')

# Rebuild
engine = ImageSearchEngine.build_from_folder(
    image_dir=IMAGE_DIR,
    index_dir=INDEX_DIR,
    index_type='flat',
    batch_size=32,
)
print(f'Rebuilt: {engine.index_size()} images.')